In [1]:
# Cell 1 — NumPy vs PyTorch tensor comparison
import numpy as np
import torch

# NumPy array — you already know this
arr = np.array([[1.0, 2.0], [3.0, 4.0]])
print("NumPy:", arr.shape, arr.dtype)

# PyTorch tensor — same data, new superpowers
t = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("Tensor:", t.shape, t.dtype)

# They look the same — the difference is under the hood
print("\nNumPy value at [0,1]:", arr[0,1])
print("Tensor value at [0,1]:", t[0,1].item())  # .item() extracts a Python float

# The key difference: requires_grad
# This tells PyTorch "track all operations on this tensor"
t_grad = torch.tensor([[1.0, 2.0], [3.0, 4.0]], requires_grad=True)
print("\nrequires_grad:", t_grad.requires_grad)  # True
2

NumPy: (2, 2) float64
Tensor: torch.Size([2, 2]) torch.float32

NumPy value at [0,1]: 2.0
Tensor value at [0,1]: 2.0

requires_grad: True


2

In [2]:
# Cell 2 — tensor shapes
# A single image: 3 colour channels, 224 pixels tall, 224 pixels wide
single_image = torch.randn(3, 224, 224)
print("Single image shape:", single_image.shape)  # torch.Size([3, 224, 224])

# A BATCH of 32 images — this is what the model processes at once
batch = torch.randn(32, 3, 224, 224)
print("Batch shape:", batch.shape)  # torch.Size([32, 3, 224, 224])
# 32 = batch size, 3 = RGB channels, 224x224 = spatial dimensions

# Your 4 OpenCV signals for a batch of 32 images
signals = torch.randn(32, 4)
print("Signals shape:", signals.shape)  # torch.Size([32, 4])
# 32 = batch size, 4 = blur/exposure/crop/metadata scores

# Shape arithmetic — always verify manually before coding
print("\nBatch dimensions explained:")
print(f"  batch[0]         = one image     shape: {batch[0].shape}")
print(f"  batch[0, 0]      = red channel   shape: {batch[0,0].shape}")
print(f"  batch[0, 0, 0]   = one row       shape: {batch[0,0,0].shape}")
print(f"  batch[0, 0, 0, 0]= one pixel     value: {batch[0,0,0,0].item():.4f}")

Single image shape: torch.Size([3, 224, 224])
Batch shape: torch.Size([32, 3, 224, 224])
Signals shape: torch.Size([32, 4])

Batch dimensions explained:
  batch[0]         = one image     shape: torch.Size([3, 224, 224])
  batch[0, 0]      = red channel   shape: torch.Size([224, 224])
  batch[0, 0, 0]   = one row       shape: torch.Size([224])
  batch[0, 0, 0, 0]= one pixel     value: -0.3976


In [3]:
# Cell 3 — device handling
# Check if a GPU is available (it won't be on your MacBook Air)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)  # Will print: cpu

# RULE: the model and its inputs must be on the SAME device
# This is the most common error in PyTorch — TypeError: device mismatch

# Move a tensor to a device
t = torch.randn(3, 224, 224)
t = t.to(device)
print("Tensor on:", t.device)

# In training: always do this for every batch
# imgs    = imgs.to(device)
# signals = signals.to(device)
# labels  = labels.to(device)

Using device: cpu
Tensor on: cpu


In [15]:
# Cell 4 — load and inspect EfficientNet-B0
import torch
from torchvision import models

# Load pretrained — weights="DEFAULT" means ImageNet pretrained
backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Print the full architecture — read through it
print(backbone)

# It has three main parts:
# 1. backbone.features — the CNN layers that extract features
# 2. backbone.avgpool  — global average pooling (compresses spatial info)
# 3. backbone.classifier — the final prediction layer (we REPLACE this)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /Users/ma2233/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100.0%


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [16]:
# Cell 5 — trace one image through EfficientNet step by step
backbone.eval()  # evaluation mode — no dropout, no batchnorm randomness

dummy = torch.randn(1, 3, 224, 224)  # one fake image

with torch.no_grad():  # don't track gradients — we're just inspecting
    # Step 1: feature extraction
    # backbone.features = 9 MBConv blocks that progressively extract patterns
    # Input: (1, 3, 224, 224) — one RGB image
    # Output: (1, 1280, 7, 7) — 1280 feature maps, each 7×7 spatial
    features = backbone.features(dummy)
    print("After features:", features.shape)  # [1, 1280, 7, 7]

    # Step 2: global average pooling
    # Collapses the 7×7 spatial dimension into a single number per channel
    # Input: (1, 1280, 7, 7) → Output: (1, 1280, 1, 1)
    pooled = backbone.avgpool(features)
    print("After avgpool:", pooled.shape)   # [1, 1280, 1, 1]

    # Step 3: flatten — turn (1, 1280, 1, 1) into (1, 1280)
    # This is the feature vector — 1280 numbers describing the image
    flat = torch.flatten(pooled, 1)
    print("After flatten:", flat.shape)    # [1, 1280]

print("\n1280 — this is the feature vector dimension")
print("This is what gets concatenated with your 4 scorer signals")

After features: torch.Size([1, 1280, 7, 7])
After avgpool: torch.Size([1, 1280, 1, 1])
After flatten: torch.Size([1, 1280])

1280 — this is the feature vector dimension
This is what gets concatenated with your 4 scorer signals


In [17]:
# Cell 6 — feature fusion
# EfficientNet gives 1280 numbers from pixels
# Your 4 scorers give 4 numbers from classical CV
# Together: 1284 numbers → richer description of the image

cnn_features = torch.randn(1, 1280)  # from EfficientNet
cv_signals   = torch.tensor([[0.9, 0.7, 0.8, 0.33]])  # blur, exposure, crop, metadata

# torch.cat joins along a dimension
# dim=1 means "join along the column dimension" (concatenate features)
fused = torch.cat([cnn_features, cv_signals], dim=1)
print("CNN features:", cnn_features.shape)    # [1, 1280]
print("CV signals:  ", cv_signals.shape)      # [1, 4]
print("Fused:       ", fused.shape)           # [1, 1284] ← feeds both heads

# This works for a whole batch too
batch_cnn = torch.randn(32, 1280)
batch_sig = torch.randn(32, 4)
batch_fused = torch.cat([batch_cnn, batch_sig], dim=1)
print("\nBatch fused:", batch_fused.shape)    # [32, 1284]

CNN features: torch.Size([1, 1280])
CV signals:   torch.Size([1, 4])
Fused:        torch.Size([1, 1284])

Batch fused: torch.Size([32, 1284])


In [18]:
# Cell 7 — what nn.Linear does
import torch.nn as nn

# nn.Linear(in_features, out_features) = one fully connected layer
# It learns a weight matrix W and bias b
# output = input @ W.T + b

layer = nn.Linear(1284, 256)
print("Weight shape:", layer.weight.shape)  # [256, 1284]
print("Bias shape  :", layer.bias.shape)    # [256]

# Pass some data through
x = torch.randn(32, 1284)  # batch of 32, 1284 features each
out = layer(x)
print("Input :", x.shape)   # [32, 1284]
print("Output:", out.shape) # [32, 256] — compressed to 256

# ReLU activation — clips negative values to 0
# Without it, stacking linear layers is the same as one linear layer
relu = nn.ReLU()
out_activated = relu(out)
print("After ReLU:", out_activated.shape)  # [32, 256] — same shape, some zeros now
print("Min value:", out_activated.min().item())  # 0.0 — ReLU removes negatives

Weight shape: torch.Size([256, 1284])
Bias shape  : torch.Size([256])
Input : torch.Size([32, 1284])
Output: torch.Size([32, 256])
After ReLU: torch.Size([32, 256])
Min value: 0.0


In [21]:
# Cell 8 — the two prediction heads

# HEAD 1: Score head (regression)
# Predicts a continuous number between 0 and 1
# Sigmoid squashes any number into [0, 1]
score_head = nn.Sequential(
    nn.Linear(1284, 256),
    nn.ReLU(),
    nn.Dropout(0.3),    # randomly zeros 30% of neurons during training → reduces overfitting
    nn.Linear(256, 1),  # compress to single number
    nn.Sigmoid()        # squash to [0, 1] range
)

# HEAD 2: Class head (classification)
# Predicts which of 5 failure types: pass/blur/exposure/crop_error/metadata_fail
# No softmax here — CrossEntropyLoss applies it internally
class_head = nn.Sequential(
    nn.Linear(1284, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 5)   # 5 output neurons, one per class
)

# Test both heads with fake fused features
fused = torch.randn(32, 1284)  # batch of 32

scores  = score_head(fused)
classes = class_head(fused)

print("Score output :", scores.shape)   # [32, 1] — one score per image
print("Class output :", classes.shape)  # [32, 5] — 5 raw logits per image
print(f"Score range  : {scores.min().item():.3f}, to {scores.max().item():.3f}")  # 0 to 1

# To get the predicted class: take argmax across the 5 class scores
predicted_class = classes.argmax(dim=1)
print("Predicted classes:", predicted_class[:5])  # e.g. tensor([2, 0, 4, 1, 3])

CLASS_NAMES = ["pass", "blur", "exposure", "crop_error", "metadata_fail"]
print("Class names:", [CLASS_NAMES[i] for i in predicted_class[:5].tolist()])

Score output : torch.Size([32, 1])
Class output : torch.Size([32, 5])
Score range  : 0.421, to 0.627
Predicted classes: tensor([0, 1, 3, 0, 0])
Class names: ['pass', 'blur', 'crop_error', 'pass', 'pass']


In [23]:
# Cell 8 — the two prediction heads

# HEAD 1: Score head (regression)
# Predicts a continuous number between 0 and 1
# Sigmoid squashes any number into [0, 1]
score_head = nn.Sequential(
    nn.Linear(1284, 256),
    nn.ReLU(),
    nn.Dropout(0.3),    # randomly zeros 30% of neurons during training → reduces overfitting
    nn.Linear(256, 1),  # compress to single number
    nn.Sigmoid()        # squash to [0, 1] range
)

# HEAD 2: Class head (classification)
# Predicts which of 5 failure types: pass/blur/exposure/crop_error/metadata_fail
# No softmax here — CrossEntropyLoss applies it internally
class_head = nn.Sequential(
    nn.Linear(1284, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 5)   # 5 output neurons, one per class
)

# Test both heads with fake fused features
fused = torch.randn(32, 1284)  # batch of 32

scores  = score_head(fused)
classes = class_head(fused)

print("Score output :", scores.shape)   # [32, 1] — one score per image
print("Class output :", classes.shape)  # [32, 5] — 5 raw logits per image
print(f"Score range  : {scores.min().item():.3f} to  {scores.max().item():.3f}")  # 0 to 1

# To get the predicted class: take argmax across the 5 class scores
predicted_class = classes.argmax(dim=1)
print("Predicted classes:", predicted_class[:5])  # e.g. tensor([2, 0, 4, 1, 3])

CLASS_NAMES = ["pass", "blur", "exposure", "crop_error", "metadata_fail"]
print("Class names:", [CLASS_NAMES[i] for i in predicted_class[:5].tolist()])

Score output : torch.Size([32, 1])
Class output : torch.Size([32, 5])
Score range  : 0.429 to  0.595
Predicted classes: tensor([0, 3, 4, 3, 1])
Class names: ['pass', 'crop_error', 'metadata_fail', 'crop_error', 'blur']


In [24]:
# Cell 9 — what nn.Module is

# Every PyTorch model inherits from nn.Module
# It provides: parameter tracking, .to(device), .eval(), .train(), saving/loading
# You define: __init__ (layers) and forward (how data flows through them)

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()              # ALWAYS call this first
        self.layer1 = nn.Linear(10, 5)  # define layers as attributes
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(5, 1)

    def forward(self, x):              # define the data flow
        x = self.layer1(x)             # x: [B, 10] → [B, 5]
        x = self.relu(x)               # x: [B, 5]  → [B, 5]
        x = self.layer2(x)             # x: [B, 5]  → [B, 1]
        return x

model = SimpleModel()
x = torch.randn(4, 10)  # batch of 4
out = model(x)           # calls forward() automatically
print("Output:", out.shape)  # [4, 1]

# Count parameters
total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total:,}")  # 61 (tiny model)

Output: torch.Size([4, 1])
Parameters: 61


In [26]:
# Cell 10 — forward pass test
import sys; sys.path.insert(0, "src")
from model import QualityGateModel
import torch

model = QualityGateModel()
model.eval()

# Fake inputs — shape matters, values don't for this test
dummy_images  = torch.randn(4, 3, 224, 224)  # batch of 4 images
dummy_signals = torch.randn(4, 4)            # batch of 4 signal vectors

with torch.no_grad():
    score, cls = model(dummy_images, dummy_signals)

print("Score output shape:", score.shape)   # torch.Size([4])
print("Class output shape:", cls.shape)     # torch.Size([4, 5])
print(f"Score range  : {scores.min().item():.3f} to  {scores.max().item():.3f}")  # 0 to 1
print("Score values:", score)  # 4 numbers between 0 and 1

# Predicted classes for this batch
CLASS_NAMES = ["pass","blur","exposure","crop_error","metadata_fail"]
for i, (s, c) in enumerate(zip(score, cls.argmax(1))):
    print(f"  Image {i}: score={s.item():.3f}  class={CLASS_NAMES[c]}")
2

Score output shape: torch.Size([4])
Class output shape: torch.Size([4, 5])
Score range  : 0.429 to  0.595
Score values: tensor([0.5048, 0.4996, 0.5038, 0.5023])
  Image 0: score=0.505  class=metadata_fail
  Image 1: score=0.500  class=metadata_fail
  Image 2: score=0.504  class=blur
  Image 3: score=0.502  class=blur


2

In [27]:
# Cell 11 — model parameter count
model = QualityGateModel()

total    = sum(p.numel() for p in model.parameters())
backbone = sum(p.numel() for p in model.features.parameters())
heads    = sum(p.numel() for p in model.score_head.parameters()) + \
           sum(p.numel() for p in model.class_head.parameters())

print(f"Total parameters  : {total:>12,}")
print(f"EfficientNet (frozen): {backbone:>10,}")
print(f"New heads (trainable): {heads:>10,}")
print(f"\nWith frozen backbone you only train {heads:,} parameters")
print(f"This is why 2000 images is enough")

Total parameters  :    4,667,010
EfficientNet (frozen):  4,007,548
New heads (trainable):    659,462

With frozen backbone you only train 659,462 parameters
This is why 2000 images is enough


In [28]:
# Cell 12 — freeze the backbone (optional but recommended for small datasets)
model = QualityGateModel()

# Freeze all EfficientNet parameters
for param in model.features.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")  # ~700K instead of 6M
# Training will be ~8x faster — each backward pass is much smaller

Trainable parameters: 659,462


In [29]:
# Run this to SEE what Dropout does
x = torch.ones(1, 10)  # a vector of 10 ones
dropout = nn.Dropout(0.3)

dropout.train()  # training mode — dropout is ACTIVE
print("Training mode:", dropout(x))  # some values zeroed randomly
print("Training mode:", dropout(x))  # different zeros each time!

dropout.eval()   # eval mode — dropout is INACTIVE
print("Eval mode:    ", dropout(x))  # all ones — no dropout at inference
print("\nDropout forces the model to not rely on any single neuron")
print("This prevents overfitting on small datasets like yours")

Training mode: tensor([[0.0000, 1.4286, 0.0000, 0.0000, 0.0000, 1.4286, 1.4286, 1.4286, 1.4286,
         1.4286]])
Training mode: tensor([[1.4286, 1.4286, 1.4286, 0.0000, 0.0000, 1.4286, 1.4286, 1.4286, 1.4286,
         1.4286]])
Eval mode:     tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])

Dropout forces the model to not rely on any single neuron
This prevents overfitting on small datasets like yours


In [31]:
# Run your actual labeled images through the untrained model
# Scores will be random (untrained) but shapes must be correct
from torchvision import transforms
from PIL import Image
import sys; sys.path.insert(0, "src")
from model import QualityGateModel
from scorer import extract_signals
from pathlib import Path

model = QualityGateModel(); model.eval()
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
CLASS_NAMES = ["pass","blur","exposure","crop_error","metadata_fail"]

for cls in ["pass","blur","exposure"]:
    for p in list(Path(f"data/labeled/{cls}").glob("*.j*"))[:2]:
        img = transform(Image.open(p).convert("RGB")).unsqueeze(0)
        sig = extract_signals(str(p))
        sig_t = torch.tensor([[sig.blur_score, sig.exposure_score,
                               sig.crop_score, sig.metadata_score]])
        with torch.no_grad():
            score, logits = model(img, sig_t)
        print(f"[true:{cls:12s}] score={score.item():.3f}  pred={CLASS_NAMES[logits.argmax(1).item()]}")
# Predictions are random — model is untrained
# But if this runs without error, your full pipeline is connected

[true:pass        ] score=0.474  pred=pass
[true:pass        ] score=0.482  pred=pass
[true:blur        ] score=0.523  pred=pass
[true:blur        ] score=0.514  pred=blur
[true:exposure    ] score=0.495  pred=blur
[true:exposure    ] score=0.506  pred=blur
